In [0]:
%sql
use catalog insurance;
use schema homecredit

In [0]:
from azure.storage.blob import BlobServiceClient
import pandas as pd
import os 
connection_string="DefaultEndpointsProtocol=https;AccountName=dbq2source;AccountKey=PXXhAoeMGDPApjyShkzLheM345zVnXduPfGO3yQT0RyCOqce4YTlt5Xi3ET7mShSXhNdvxi/l5yt+AStfnKUVw==;EndpointSuffix=core.windows.net"
container_name = "source"
blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(container_name)
for blob in container_client.list_blobs():
    print(blob.name)


import io

for blob in container_client.list_blobs():
    table_name = (
        blob.name  
        .replace(".csv", "")
        .replace(".parquet", "")
        .replace("-", "_")
        .lower()
    )



    data = container_client.get_blob_client(blob.name).download_blob().readall()
    df = pd.read_csv(io.BytesIO(data))
    df = spark.createDataFrame(df)
    df.write \
            .format("delta") \
            .mode("overwrite") \
        .saveAsTable(f"insurance.homecredit.{table_name}")



POS_CASH_balance.csv
application_test.csv
application_train.csv
bureau.csv
bureau_balance.csv
credit_card_balance.csv
installments_payments.csv
previous_application.csv


In [0]:
### end-to-end data validation for Home Credit tables

from pyspark.sql import functions as F
from pyspark.sql import Row
import pandas as pd
import json
import uuid
from datetime import datetime

validation_tables = [
    "insurance.homecredit.application_test",
    "insurance.homecredit.application_train",
    "insurance.homecredit.bureau",
    "insurance.homecredit.bureau_balance",
    "insurance.homecredit.credit_card_balance",
    "insurance.homecredit.installments_payments",
    "insurance.homecredit.pos_cash_balance",
    "insurance.homecredit.previous_application",
]

dfs = {table.split('.')[-1]: spark.table(table) for table in validation_tables}

connection_string = "DefaultEndpointsProtocol=https;AccountName=dbq2source;AccountKey=qOuObNKMQMWaInvAv+97zmvZovLKnWyApY1ovXwft3L7JNOftXFCGPqoISRraFhKNBE1sEkW88a3+AStxn/v3A==;EndpointSuffix=core.windows.net"
container_name = "landing"

run_id = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
blob_base_path = f"validation_logs/homecredit/run_id={run_id}"

application_keys_df = (
    dfs["application_train"].select("SK_ID_CURR")
    .unionByName(dfs["application_test"].select("SK_ID_CURR"))
    .dropDuplicates()
)

print(f"Validation run id: {run_id}")
print(f"Validation output path in blob: {blob_base_path}")
display(spark.createDataFrame([(t,) for t in validation_tables], ["table_name"]))

/home/spark-f959de67-2d37-4f0a-9afb-23/.ipykernel/332/command-6880830979953782-1852345876:26: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_id = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]


Validation run id: 20260627T154925Z_9c3cde28
Validation output path in blob: validation_logs/homecredit/run_id=20260627T154925Z_9c3cde28


table_name
insurance.homecredit.application_test
insurance.homecredit.application_train
insurance.homecredit.bureau
insurance.homecredit.bureau_balance
insurance.homecredit.credit_card_balance
insurance.homecredit.installments_payments
insurance.homecredit.pos_cash_balance
insurance.homecredit.previous_application


30 data validations included in this notebook:

* 1. `application_train` row count is greater than 0
* 2. `application_test` row count is greater than 0
* 3. `bureau` row count is greater than 0
* 4. `bureau_balance` row count is greater than 0
* 5. `credit_card_balance` row count is greater than 0
* 6. `installments_payments` row count is greater than 0
* 7. `pos_cash_balance` row count is greater than 0
* 8. `previous_application` row count is greater than 0
* 9. `sample_submission` row count is greater than 0
* 10. `application_train.SK_ID_CURR` is not null
* 11. `application_test.SK_ID_CURR` is not null
* 12. `bureau.SK_ID_BUREAU` is not null
* 13. `bureau_balance.SK_ID_BUREAU` is not null
* 14. `previous_application.SK_ID_PREV` is not null
* 15. `sample_submission.SK_ID_CURR` is not null
* 16. `application_train.SK_ID_CURR` is unique
* 17. `application_test.SK_ID_CURR` is unique
* 18. `bureau.SK_ID_BUREAU` is unique
* 19. `sample_submission.SK_ID_CURR` is unique
* 20. `application_train.TARGET` is only 0 or 1
* 21. `sample_submission.TARGET` is between 0 and 1
* 22. `application_train.AMT_CREDIT` is positive when present
* 23. `application_train.AMT_INCOME_TOTAL` is positive when present
* 24. `application_train.DAYS_BIRTH` is negative when present
* 25. `application_train.application_date` is parseable to date when present
* 26. `bureau_balance.STATUS` is in the allowed domain
* 27. `bureau.SK_ID_CURR` resolves to the application universe
* 28. `bureau_balance.SK_ID_BUREAU` resolves to `bureau`
* 29. `previous_application.SK_ID_CURR` resolves to the application universe
* 30. Downstream `SK_ID_PREV` keys in `credit_card_balance`, `installments_payments`, and `pos_cash_balance` resolve to `previous_application`

In [0]:
from typing import Callable, Dict, List

allowed_bureau_balance_status = ['0', '1', '2', '3', '4', '5', 'C', 'X']

validation_results = []


def add_result(validation_id, validation_name, table_name, category, passed, failed_records, checked_records, details):
    validation_results.append(
        Row(
            run_id=run_id,
            validation_id=validation_id,
            validation_name=validation_name,
            table_name=table_name,
            category=category,
            passed=bool(passed),
            failed_records=int(failed_records),
            checked_records=int(checked_records),
            failure_pct=float((failed_records / checked_records) if checked_records else 0.0),
            details=json.dumps(details, default=str),
            validated_at_utc=datetime.utcnow().isoformat()
        )
    )


def row_count_check(validation_id, table_name):
    checked_records = dfs[table_name].count()
    add_result(
        validation_id,
        f"{table_name} row count > 0",
        table_name,
        "volume",
        checked_records > 0,
        0 if checked_records > 0 else 1,
        max(checked_records, 1),
        {"row_count": checked_records}
    )


def not_null_check(validation_id, table_name, column_name):
    df = dfs[table_name]
    checked_records = df.count()
    failed_records = df.filter(F.col(column_name).isNull()).count()
    add_result(
        validation_id,
        f"{table_name}.{column_name} has no nulls",
        table_name,
        "completeness",
        failed_records == 0,
        failed_records,
        checked_records,
        {"column_name": column_name}
    )


def unique_check(validation_id, table_name, column_name):
    df = dfs[table_name]
    checked_records = df.count()
    distinct_records = df.select(column_name).distinct().count()
    failed_records = checked_records - distinct_records
    add_result(
        validation_id,
        f"{table_name}.{column_name} is unique",
        table_name,
        "uniqueness",
        failed_records == 0,
        failed_records,
        checked_records,
        {"column_name": column_name, "distinct_records": distinct_records}
    )


def allowed_values_check(validation_id, table_name, column_name, allowed_values):
    df = dfs[table_name]
    checked_records = df.filter(F.col(column_name).isNotNull()).count()
    failed_records = df.filter(F.col(column_name).isNotNull() & (~F.col(column_name).isin(allowed_values))).count()
    invalid_examples = [row[column_name] for row in df.filter(F.col(column_name).isNotNull() & (~F.col(column_name).isin(allowed_values))).select(column_name).distinct().limit(10).collect()]
    add_result(
        validation_id,
        f"{table_name}.{column_name} values are allowed",
        table_name,
        "domain",
        failed_records == 0,
        failed_records,
        max(checked_records, 1),
        {"column_name": column_name, "allowed_values": allowed_values, "invalid_examples": invalid_examples}
    )


def range_check(validation_id, table_name, column_name, predicate_description, predicate):
    df = dfs[table_name]
    checked_df = df.filter(F.col(column_name).isNotNull())
    checked_records = checked_df.count()
    failed_records = checked_df.filter(~predicate(F.col(column_name))).count()
    stats = checked_df.agg(
        F.min(F.col(column_name)).alias('min_value'),
        F.max(F.col(column_name)).alias('max_value')
    ).collect()[0]
    add_result(
        validation_id,
        f"{table_name}.{column_name} {predicate_description}",
        table_name,
        "range",
        failed_records == 0,
        failed_records,
        max(checked_records, 1),
        {"column_name": column_name, "rule": predicate_description, "min_value": stats['min_value'], "max_value": stats['max_value']}
    )


def date_parse_check(validation_id, table_name, column_name):
    df = dfs[table_name]
    checked_df = df.filter(F.col(column_name).isNotNull())
    checked_records = checked_df.count()
    parsed_col = F.coalesce(
        F.to_date(F.col(column_name), 'yyyy-MM-dd'),
        F.to_date(F.col(column_name), 'MM/dd/yyyy'),
        F.to_date(F.col(column_name), 'dd-MM-yyyy')
    )
    failed_records = checked_df.filter(parsed_col.isNull()).count()
    sample_bad_values = [row[column_name] for row in checked_df.filter(parsed_col.isNull()).select(column_name).distinct().limit(10).collect()]
    add_result(
        validation_id,
        f"{table_name}.{column_name} is parseable as a date",
        table_name,
        "format",
        failed_records == 0,
        failed_records,
        max(checked_records, 1),
        {"column_name": column_name, "sample_bad_values": sample_bad_values}
    )


def foreign_key_check(validation_id, child_table_name, child_column, parent_df, parent_column, check_name):
    child_df = dfs[child_table_name].select(child_column).filter(F.col(child_column).isNotNull())
    checked_records = child_df.count()
    failed_records = (
        child_df.alias('child')
        .join(parent_df.select(parent_column).dropDuplicates().alias('parent'), F.col(f'child.{child_column}') == F.col(f'parent.{parent_column}'), 'left_anti')
        .count()
    )
    add_result(
        validation_id,
        check_name,
        child_table_name,
        "referential_integrity",
        failed_records == 0,
        failed_records,
        max(checked_records, 1),
        {"child_column": child_column, "parent_column": parent_column}
    )


In [0]:
row_count_check(1, 'application_train')
row_count_check(2, 'application_test')
row_count_check(3, 'bureau')
row_count_check(4, 'bureau_balance')
row_count_check(5, 'credit_card_balance')
row_count_check(6, 'installments_payments')
row_count_check(7, 'pos_cash_balance')
row_count_check(8, 'previous_application')


not_null_check(9, 'application_train', 'SK_ID_CURR')
not_null_check(10, 'application_test', 'SK_ID_CURR')
not_null_check(11, 'bureau', 'SK_ID_BUREAU')
not_null_check(12, 'bureau_balance', 'SK_ID_BUREAU')
not_null_check(13, 'previous_application', 'SK_ID_PREV')


unique_check(14, 'application_train', 'SK_ID_CURR')
unique_check(15, 'application_test', 'SK_ID_CURR')
unique_check(16, 'bureau', 'SK_ID_BUREAU')


allowed_values_check(17, 'application_train', 'TARGET', [0.0, 1.0])
# range_check(18, 'sample_submission', 'TARGET', 'is between 0 and 1 inclusive', lambda c: (c >= F.lit(0.0)) & (c <= F.lit(1.0)))
range_check(18, 'application_train', 'AMT_CREDIT', 'is positive when present', lambda c: c > F.lit(0))
range_check(19, 'application_train', 'AMT_INCOME_TOTAL', 'is positive when present', lambda c: c > F.lit(0))
range_check(20, 'application_train', 'DAYS_BIRTH', 'is negative when present', lambda c: c < F.lit(0))
# date_parse_check(21, 'application_train', 'application_date')
allowed_values_check(21, 'bureau_balance', 'STATUS', allowed_bureau_balance_status)

foreign_key_check(
    22,
    'bureau',
    'SK_ID_CURR',
    application_keys_df,
    'SK_ID_CURR',
    'bureau.SK_ID_CURR exists in application_train or application_test'
)
foreign_key_check(
    23,
    'bureau_balance',
    'SK_ID_BUREAU',
    dfs['bureau'],
    'SK_ID_BUREAU',
    'bureau_balance.SK_ID_BUREAU exists in bureau'
)
foreign_key_check(
    24,
    'previous_application',
    'SK_ID_CURR',
    application_keys_df,
    'SK_ID_CURR',
    'previous_application.SK_ID_CURR exists in application_train or application_test'
)

sk_prev_children = (
    dfs['credit_card_balance'].select('SK_ID_PREV').filter(F.col('SK_ID_PREV').isNotNull())
    .unionByName(dfs['installments_payments'].select('SK_ID_PREV').filter(F.col('SK_ID_PREV').isNotNull()))
    .unionByName(dfs['pos_cash_balance'].select('SK_ID_PREV').filter(F.col('SK_ID_PREV').isNotNull()))
)
failed_sk_prev = (
    sk_prev_children.alias('child')
    .join(dfs['previous_application'].select('SK_ID_PREV').dropDuplicates().alias('parent'), 'SK_ID_PREV', 'left_anti')
    .count()
)
checked_sk_prev = sk_prev_children.count()
add_result(
    25,
    'Downstream SK_ID_PREV values exist in previous_application',
    'credit_card_balance/installments_payments/pos_cash_balance',
    'referential_integrity',
    failed_sk_prev == 0,
    failed_sk_prev,
    max(checked_sk_prev, 1),
    {'child_tables': ['credit_card_balance', 'installments_payments', 'pos_cash_balance'], 'parent_table': 'previous_application'}
)

validation_results_df = spark.createDataFrame(validation_results).orderBy('validation_id')
validation_summary_df = validation_results_df.groupBy('run_id').agg(
    F.count('*').alias('total_validations'),
    F.sum(F.when(F.col('passed'), 1).otherwise(0)).alias('passed_validations'),
    F.sum(F.when(~F.col('passed'), 1).otherwise(0)).alias('failed_validations')
).withColumn('blob_base_path', F.lit(blob_base_path))

failed_validations_df = validation_results_df.filter(~F.col('passed')).orderBy('validation_id')

print('Detailed validation results:')
display(validation_results_df)

print('Validation summary:')
display(validation_summary_df)

print('Failed validations only:')
display(failed_validations_df)

/home/spark-f959de67-2d37-4f0a-9afb-23/.ipykernel/332/command-6880830979953784-3938239583:21: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  validated_at_utc=datetime.utcnow().isoformat()


Detailed validation results:


run_id,validation_id,validation_name,table_name,category,passed,failed_records,checked_records,failure_pct,details,validated_at_utc
20260627T154925Z_9c3cde28,1,application_train row count > 0,application_train,volume,true,0,307511,0.0,"{""row_count"": 307511}",2026-06-27T15:49:52.634638
20260627T154925Z_9c3cde28,2,application_test row count > 0,application_test,volume,true,0,48744,0.0,"{""row_count"": 48744}",2026-06-27T15:49:52.980359
20260627T154925Z_9c3cde28,3,bureau row count > 0,bureau,volume,true,0,1716428,0.0,"{""row_count"": 1716428}",2026-06-27T15:49:53.275252
20260627T154925Z_9c3cde28,4,bureau_balance row count > 0,bureau_balance,volume,true,0,27299925,0.0,"{""row_count"": 27299925}",2026-06-27T15:49:53.559611
20260627T154925Z_9c3cde28,5,credit_card_balance row count > 0,credit_card_balance,volume,true,0,3840312,0.0,"{""row_count"": 3840312}",2026-06-27T15:49:53.840626
20260627T154925Z_9c3cde28,6,installments_payments row count > 0,installments_payments,volume,true,0,13605401,0.0,"{""row_count"": 13605401}",2026-06-27T15:49:54.125973
20260627T154925Z_9c3cde28,7,pos_cash_balance row count > 0,pos_cash_balance,volume,true,0,10001358,0.0,"{""row_count"": 10001358}",2026-06-27T15:49:54.405221
20260627T154925Z_9c3cde28,8,previous_application row count > 0,previous_application,volume,true,0,1670214,0.0,"{""row_count"": 1670214}",2026-06-27T15:49:54.698699
20260627T154925Z_9c3cde28,9,application_train.SK_ID_CURR has no nulls,application_train,completeness,true,0,307511,0.0,"{""column_name"": ""SK_ID_CURR""}",2026-06-27T15:49:55.861964
20260627T154925Z_9c3cde28,10,application_test.SK_ID_CURR has no nulls,application_test,completeness,true,0,48744,0.0,"{""column_name"": ""SK_ID_CURR""}",2026-06-27T15:49:56.597118


Validation summary:


run_id,total_validations,passed_validations,failed_validations,blob_base_path
20260627T154925Z_9c3cde28,25,23,2,validation_logs/homecredit/run_id=20260627T154925Z_9c3cde28


Failed validations only:


run_id,validation_id,validation_name,table_name,category,passed,failed_records,checked_records,failure_pct,details,validated_at_utc
20260627T154925Z_9c3cde28,23,bureau_balance.SK_ID_BUREAU exists in bureau,bureau_balance,referential_integrity,false,3120184,27299925,0.11429276820357565,"{""child_column"": ""SK_ID_BUREAU"", ""parent_column"": ""SK_ID_BUREAU""}",2026-06-27T15:50:12.156082
20260627T154925Z_9c3cde28,25,Downstream SK_ID_PREV values exist in previous_application,credit_card_balance/installments_payments/pos_cash_balance,referential_integrity,false,2674203,27447071,0.09743127053520574,"{""child_tables"": [""credit_card_balance"", ""installments_payments"", ""pos_cash_balance""], ""parent_table"": ""previous_application""}",2026-06-27T15:50:16.880586


In [0]:
validation_results_table = 'insurance.homecredit.validation_results'
validation_summary_table = 'insurance.homecredit.validation_summary'
validation_failed_table = 'insurance.homecredit.validation_failed_results'

validation_results_df.write.format('delta').mode('append').saveAsTable(validation_results_table)
validation_summary_df.write.format('delta').mode('append').saveAsTable(validation_summary_table)
failed_validations_df.write.format('delta').mode('append').saveAsTable(validation_failed_table)

artifact_rows = [
    ('run_id', run_id),
    ('blob_base_path', blob_base_path),
    ('results_table', validation_results_table),
    ('summary_table', validation_summary_table),
    ('failed_table', validation_failed_table)
]

artifacts_df = spark.createDataFrame(artifact_rows, ['artifact_type', 'artifact_location'])
display(artifacts_df)


artifact_type,artifact_location
run_id,20260627T154925Z_9c3cde28
blob_base_path,validation_logs/homecredit/run_id=20260627T154925Z_9c3cde28
results_table,insurance.homecredit.validation_results
summary_table,insurance.homecredit.validation_summary
failed_table,insurance.homecredit.validation_failed_results
